# Roosterly: run with Kaggle or Groq
Choose one provider and one run mode in the first cell. `evaluate` and `dry_run` never post ticket actions; `live` processes open tickets. The private repository is loaded with a read-only deploy key from Kaggle Secrets. Keep this notebook private.


In [ ]:
PROVIDER = 'kaggle'  # 'kaggle' uses the GPU; 'groq' uses the API
RUN_MODE = 'evaluate'  # 'evaluate', 'dry_run', 'status', or 'live'
assert PROVIDER in ('kaggle', 'groq')
assert RUN_MODE in ('evaluate', 'dry_run', 'status', 'live')
import base64, os, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient
repo = Path('/kaggle/working/Roosterly')
deploy_key = UserSecretsClient().get_secret('GITHUB_DEPLOY_KEY')
key_path = Path('/kaggle/working/roosterly_deploy_key')
host_path = Path('/kaggle/working/github_known_hosts')
key_path.write_bytes(base64.b64decode(deploy_key))
key_path.chmod(0o600)
host_path.write_text('github.com ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIOMqqnkVzrm0SdG6UOoqKLsabgH5C9okWi0dh2l9GKJl\n')
ssh_env = os.environ.copy()
ssh_env['GIT_SSH_COMMAND'] = f'ssh -i {key_path} -o IdentitiesOnly=yes -o UserKnownHostsFile={host_path} -o StrictHostKeyChecking=yes'
try:
    if (repo / '.git').exists():
        subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'kaggle-gpu-inference'], check=True, env=ssh_env)
    else:
        subprocess.run(['git', 'clone', '--branch', 'kaggle-gpu-inference', '--single-branch', 'git@github.com:himanrawat/Roosterly.git', str(repo)], check=True, env=ssh_env)
finally:
    key_path.unlink(missing_ok=True)
    del deploy_key, ssh_env
requirements = 'requirements-kaggle.txt' if PROVIDER == 'kaggle' else 'requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / requirements)], check=True)
print('Repository and dependencies ready')


In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ('HELPDESK_API_KEY', 'CANDIDATE_EMAIL'):
    os.environ[name] = secrets.get_secret(name)
os.environ['AGENT_PROVIDER'] = 'local' if PROVIDER == 'kaggle' else 'groq'
if PROVIDER == 'kaggle':
    import torch
    assert torch.cuda.is_available(), 'Enable GPU T4 x2 in Session options first'
    print('GPU:', torch.cuda.get_device_name(0))
    os.environ['LOCAL_MODEL'] = 'Qwen/Qwen3-8B'
else:
    os.environ['GROQ_API_KEY'] = secrets.get_secret('GROQ_API_KEY')
print('Provider:', PROVIDER, '| Run mode:', RUN_MODE, '| Required secrets loaded')


In [ ]:
from datetime import datetime, timezone
if RUN_MODE == 'evaluate':
    label = f'{PROVIDER}_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}'
    command = [sys.executable, '-m', 'agent.evaluate', '--label', label, '--runs', '1']
elif RUN_MODE == 'dry_run':
    command = [sys.executable, '-m', 'agent', '--dry-run']
elif RUN_MODE == 'status':
    command = [sys.executable, '-m', 'agent', '--status']
else:
    print('LIVE RUN: posting decisions for every open helpdesk ticket')
    command = [sys.executable, '-m', 'agent']
result = subprocess.run(command, cwd=repo, env=os.environ.copy())
print('Run exit code:', result.returncode)
if result.returncode:
    raise RuntimeError(f'Run failed with exit code {result.returncode}')


## Run modes
Change only `PROVIDER` and `RUN_MODE` in the first cell, then run all cells. `evaluate` scores the labeled examples without posting; `dry_run` processes open tickets without posting; `status` shows counts; `live` posts one action for each open ticket. Keep the notebook private because it has access to helpdesk secrets. Do not reset the existing scored practice run just to test Kaggle.
